# Phase 1 sources and provenance
This notebook inventories checked files without network retries. The normal-terminal runner's
`--acquire` mode downloads missing sources after the early developer sample.
Historical NOAA **LCDv1** is explicit; v2 is not substituted. CSV clocks must be checked against raw
UTC station reports before conversion. Raw/validated outputs are preserved.
Default mode is a clearly labelled sample. `run_phase1.py` gates execution on tests and Ruff.
Last successful full Phase 1 execution: 15 November 2026 IST; finalized version `finalized-20261114T190620-6b22f8cb`. See shared handover for verification and reuse.


In [1]:
# Phase 1: locate the repo and record the execution environment.
import os
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    raise RuntimeError("Run from the repository root or notebooks folder")
sys.path.insert(0, str(ROOT))
from notebooks.run_phase1 import runtime_versions, run_preflight  # noqa: E402
print(runtime_versions())
if os.environ.get("ADS_PREFLIGHT_COMPLETE") != "1":
    print("Validation receipt:", run_preflight(ROOT))
MODE = os.environ.get("ADS_PHASE1_MODE", "sample")
if MODE not in ["sample", "available"]:
    raise ValueError("ADS_PHASE1_MODE must be sample or available")


{'python': '3.11.9', 'java': 'openjdk version "21.0.11" 2026-04-21', 'pyspark': '4.0.1', 'py4j': '0.10.9.9', 'pyarrow': '24.0.0', 'pyshp': '2.3.1', 'shapely': '2.1.2', 'pyproj': '3.7.2', 'ruff': '0.14.4', 'pytest': '9.1.1', 'nbclient': '0.10.4'}


## Inventory and file checks
Checksums establish local identity, not a publisher-issued signature. Parquet row counts come
from footers; a checked file is not automatically a complete event-hour. December 2023 and
February 2025 sources are guards only; analytical periods are event-based.

In [2]:
import json
from src.phase1_sources import expected_sources, inventory_sources
inventory_path = os.environ.get("ADS_PHASE1_INVENTORY_PATH")
inventory = json.loads(Path(inventory_path).read_text()) if inventory_path else inventory_sources(expected_sources(ROOT))
print({status: sum(row["status"] == status for row in inventory)
       for status in sorted({row["status"] for row in inventory})})
for row in inventory:
    print(row["source_id"], row["status"], row.get("rows", ""), row.get("product", ""))

{'checked': 24}
trips_2023-12 checked 20516297 NYC TLC HVFHV
trips_2024-01 checked 19663930 NYC TLC HVFHV
trips_2024-02 checked 19359148 NYC TLC HVFHV
trips_2024-03 checked 21280788 NYC TLC HVFHV
trips_2024-04 checked 19733038 NYC TLC HVFHV
trips_2024-05 checked 20704538 NYC TLC HVFHV
trips_2024-06 checked 20123226 NYC TLC HVFHV
trips_2024-07 checked 19182934 NYC TLC HVFHV
trips_2024-08 checked 19128392 NYC TLC HVFHV
trips_2024-09 checked 19209788 NYC TLC HVFHV
trips_2024-10 checked 20028282 NYC TLC HVFHV
trips_2024-11 checked 19987533 NYC TLC HVFHV
trips_2024-12 checked 21068851 NYC TLC HVFHV
trips_2025-01 checked 20405666 NYC TLC HVFHV
trips_2025-02 checked 19339461 NYC TLC HVFHV
taxi_zone_lookup.csv checked  TLC zones
taxi_zones.zip checked  TLC zones
lcdv1_central_park_2024 checked  NOAA LCDv1 historical CSV
lcdv1_laguardia_2024 checked  NOAA LCDv1 historical CSV
lcdv1_jfk_2024 checked  NOAA LCDv1 historical CSV
lcdv1_central_park_2025 checked  NOAA LCDv1 historical CSV
lcdv1_lagua

## Zone topology and units
Read the original geometry without repair. Rook/shared-boundary and point-touch neighbours are
separate fields. Centroid distance is a geometric proxy, not observed driving distance.

In [3]:
from src.phase1_sources import build_zone_metadata
zone_rows, edge_rows = build_zone_metadata(
    ROOT / "data/raw/zones/taxi_zone_lookup.csv",
    ROOT / "data/raw/zones/taxi_zones/taxi_zones.shp",
)
assert len(zone_rows) == 263
assert len({row["zone_id"] for row in zone_rows}) == 263
print("Verified zones:", len(zone_rows), "directed neighbour edges:", len(edge_rows))
print("Source CRS:", zone_rows[0]["source_crs"])
print("Raw files were not changed. Weather/calendar readiness is verified during integration.")

Verified zones: 263 directed neighbour edges: 692
Source CRS: EPSG:2263
Raw files were not changed. Weather/calendar readiness is verified during integration.
